[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-kmeans.ipynb)

# K-Means Clustering

*AIBits Academy · Machine Learning End To End · Unsupervised Learning*

A partition-based clustering algorithm that iteratively assigns points to the nearest centroid and updates centroids — discovering natural groupings without labels.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

> **📋 Real-World Case Study — Online Shopper Purchasing Intention**
>
> An e-commerce team wants to understand visitor behaviour patterns — session duration, pages visited, bounce rate, time-of-year — without any pre-defined labels for "browser" vs "researcher" vs "ready-to-buy." K-Means naturally surfaces these behavioural segments directly from session data, which can then feed a downstream supervised model (or a targeted UX intervention) — exploratory clustering as a first step before any labelled prediction task even begins.

## The Algorithm

**K-Means Steps:** 
 1. Initialise k centroids (randomly or K-Means++) 
 2. **Assignment:** Assign each point to the nearest centroid (Euclidean distance) 
 3. **Update:** Move each centroid to the mean of its assigned points 
 4. Repeat steps 2–3 until centroids stop moving (convergence) or max iterations reached 
**Objective:** Minimise Within-Cluster Sum of Squares (WCSS): Σₖ Σᵢ∈Cₖ ‖xᵢ − μₖ‖²

> **📊 Prerequisite refresher**
>
> The Euclidean distance in Step 2, and the squared ‖xᵢ − μₖ‖² in the WCSS objective, are both the L2 norm introduced on the **Linear Algebra for ML** prerequisite page — the same norm behind SVM's margin and Ridge's penalty. K-Means is, at its core, "group points to minimise total squared L2 distance to a representative centroid per group," with no new distance concept required.

## Animated K-Means Convergence

Watch Lloyd's algorithm cluster Mumbai neighbourhood data (rent ₹ vs distance to CBD km) into 3 groups — starting from a *deliberately bad* initialisation with all three centroids dropped in the suburbs corner, so you can see the full convergence journey. Each iteration has two visible phases, matching the algorithm exactly: **Assign** (every point takes the colour of its nearest centroid — thin lines show the assignments) and **Update** (each centroid jumps to the mean of its assigned points — dotted trails show the path travelled). The WCSS readout drops monotonically until the centroids stop moving.

> **📏 Distance is measured on scaled axes**
>
> The two features live on very different scales — distance to CBD spans about 0–20 km, but rent spans roughly ₹28,000–95,000/month. If "nearest" were computed on the raw numbers, the rent difference (far larger in magnitude) would completely dominate, and a point could get assigned to a centroid that sits visibly far away on the plot. So this widget scales each axis before measuring distance — exactly the feature-scaling step from the page — which is why the nearest centroid you *see* is always the one a point is actually assigned to.

> **🗺 Why K-Means Regions Are Always Convex Polygons**
>
> Toggle **Show Voronoi Regions** above, then click **Half-step** or **Run K-Means** — the shaded regions update after every centroid move, so you can watch the tessellation reshape itself in real time. The boundary between any two centroids' regions is always a straight line: it is the **perpendicular bisector** of the segment joining them, because that's the exact set of points equidistant from both under Euclidean distance. With k centroids, each region is the intersection of k−1 such half-planes — a **convex polygon**, by construction. This is exactly why K-Means clusters are always described as "spherical": the algorithm's nearest-centroid rule geometrically cannot produce a concave, ring-shaped, or elongated cluster boundary, no matter how the true data is actually shaped — see the Chennai Swiggy-zones Q&A below for what goes wrong when real clusters violate this assumption.

## Choosing K — The Elbow Method

In [ ]:
from sklearn.cluster import KMeans
import numpy as np

# Mumbai neighbourhood data
np.random.seed(42)
X=np.vstack([np.random.normal([2,85],[0.7,6],(40,2)),
             np.random.normal([15,32],[2,4],(40,2)),
             np.random.normal([8,53],[1.2,5],(40,2))])

wcss=[]; silhouettes=[]
from sklearn.metrics import silhouette_score

for k in range(2,9):
    km=KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels=km.fit_predict(X)
    wcss.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))
    print(f"k={k}  WCSS={km.inertia_:8.1f}  Silhouette={silhouette_score(X,labels):.3f}")

best_k=np.argmax(silhouettes)+2
print(f"\nBest k by silhouette: {best_k}")

## K-Means Limitations and Alternatives

| Limitation | Alternative |
|---|---|
| Assumes spherical clusters | DBSCAN (density-based), GMM |
| Must specify k | DBSCAN, hierarchical clustering |
| Sensitive to outliers | K-Medoids (PAM) |
| Sensitive to initialisation | K-Means++ (sklearn default) |
| Only Euclidean distance | K-Medoids with custom distance |

> **💡 Going Deeper**
>
> Two of the alternatives above — DBSCAN and hierarchical clustering — get a full dedicated treatment (algorithms, linkage math, dendrograms, and outlier detection) in the next chapter.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Cluster three neighbourhoods

Run `KMeans(n_clusters=3, n_init=10, random_state=0)` on `X`. Store the sorted cluster sizes in `sizes` and the inertia in `inertia`.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
X, _ = make_blobs(n_samples=150, centers=[[0, 0], [8, 0], [4, 7]], cluster_std=0.6, random_state=0)
sizes = inertia = None   # TODO


In [ ]:
try:
    check("three equal clusters", sizes == [50, 50, 50])
    check("tight clusters have low inertia", inertia < 200)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
X, _ = make_blobs(n_samples=150, centers=[[0, 0], [8, 0], [4, 7]], cluster_std=0.6, random_state=0)
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
sizes = sorted(int((km.labels_ == c).sum()) for c in range(3))
inertia = km.inertia_

```

</details>

### Exercise 2 · Medium · Choose k by silhouette

For k = 2–6 compute the silhouette score and store the best k in `best_k` (the data has three real groups).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
best_k = None   # TODO (reuse X)


In [ ]:
try:
    check("silhouette picks 3", best_k == 3)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
scores = {k: silhouette_score(X, KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)) for k in range(2, 7)}
best_k = max(scores, key=scores.get)

```

</details>

### Exercise 3 · Stretch · One Lloyd iteration by hand

Write `lloyd_step(X, centers)`: assign every point to its nearest centre, then return the **new centres** (the mean of each cluster's points).

In [ ]:
import numpy as np
def lloyd_step(X, centers):
    pass   # TODO


In [ ]:
try:
    X = np.array([[0.0, 0], [1, 0], [10, 0], [11, 0]])
    new = lloyd_step(X, np.array([[0.0, 0], [10.0, 0]]))
    check("centres move to cluster means", np.allclose(new, [[0.5, 0], [10.5, 0]]))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def lloyd_step(X, centers):
    d = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
    labels = d.argmin(axis=1)
    return np.array([X[labels == k].mean(axis=0) for k in range(len(centers))])

```

K-means is just this step repeated until the centres stop moving.

</details>

---
*Back to the course: **Machine Learning End To End → K-Means Clustering**.*